# Task 3 — Time Series: Monthly Installs by Category

Monthly total installs by category. Filters: app name does not start X/Y/Z; category begins E/C/B; reviews > 500; app name excludes S. Shade month-over-month growth >20%. Requested translations are included. Chart is always visible (no time restriction). Historical install data requires a real date/month field.

**How to run:** Keep `googleplaystore.csv` and (for Task 1) `User_Reviews_Google_Play.csv` in the same folder as this notebook. Run cells from top to bottom.

**Note:** Jupyter checks the time when the task cell is executed. A continuously refreshing dashboard is needed to hide/show charts automatically without rerunning a cell.

## 1. Load and prepare the existing Google Play Store training data

This notebook uses the same training-project dataset, not a new unrelated dataset.

In [1]:
from pathlib import Path
from datetime import datetime, time
from zoneinfo import ZoneInfo
import re
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from IPython.display import display, Markdown

# All task windows use India Standard Time.
IST = ZoneInfo("Asia/Kolkata")

def in_ist_window(start_hour, end_hour):
    now = datetime.now(IST).time()
    return time(start_hour, 0) <= now < time(end_hour, 0)

def show_time_message(start_hour, end_hour):
    now = datetime.now(IST).strftime("%I:%M %p IST")
    display(Markdown(
        f"**Chart hidden:** available only from {start_hour:02d}:00 to "
        f"{end_hour:02d}:00 IST. Current time: {now}."
    ))

def find_csv(candidates):
    for candidate in candidates:
        path = Path(candidate)
        if path.exists():
            return path
    for root in [Path.cwd(), *[p for p in Path.cwd().iterdir() if p.is_dir()]]:
        for candidate in candidates:
            hits = list(root.glob(candidate))
            if hits:
                return hits[0]
    return None

def numeric_from_text(series):
    return pd.to_numeric(
        series.astype(str).str.replace(r"[^0-9.]", "", regex=True).replace("", np.nan),
        errors="coerce"
    )

def size_to_mb(value):
    s = str(value).strip()
    if not s or s.lower() in {"nan", "varies with device"}:
        return np.nan
    try:
        if s.lower().endswith("k"):
            return float(s[:-1]) / 1024
        if s.lower().endswith("m"):
            return float(s[:-1])
        return float(s)
    except ValueError:
        return np.nan

apps_path = find_csv(["googleplaystore.csv", "Google-Playstore.csv", "google_play_store.csv"])
reviews_path = find_csv(["User_Reviews_Google_Play.csv", "googleplaystore_user_reviews.csv"])

if apps_path is None:
    raise FileNotFoundError(
        "Apps CSV not found. Place googleplaystore.csv in the same folder as this notebook."
    )

apps_raw = pd.read_csv(apps_path)
reviews_raw = pd.read_csv(reviews_path) if reviews_path else pd.DataFrame()
apps = apps_raw.copy()
apps.columns = [str(c).strip() for c in apps.columns]

for col in ["Rating", "Reviews", "Installs", "Price"]:
    if col in apps.columns:
        apps[col] = numeric_from_text(apps[col])
if "Size" in apps.columns:
    apps["Size_MB"] = apps["Size"].apply(size_to_mb)
if "Category" in apps.columns:
    apps["Category"] = apps["Category"].astype(str).str.strip().str.upper()
if "App" in apps.columns:
    apps["App"] = apps["App"].astype(str).str.strip()
if "Type" in apps.columns:
    apps["Type"] = apps["Type"].astype(str).str.strip().str.title()
if "Last Updated" in apps.columns:
    apps["Last Updated"] = pd.to_datetime(apps["Last Updated"], errors="coerce")

# Standard Play Store installs are values such as "1,000,000+"; numeric_from_text parses them.
if "Installs" in apps.columns:
    apps["Installs"] = pd.to_numeric(apps["Installs"], errors="coerce")

# Merge average review sentiment subjectivity when available.
if not reviews_raw.empty and "App" in reviews_raw.columns:
    reviews = reviews_raw.copy()
    reviews.columns = [str(c).strip() for c in reviews.columns]
    if "Sentiment_Subjectivity" in reviews.columns:
        reviews["Sentiment_Subjectivity"] = pd.to_numeric(reviews["Sentiment_Subjectivity"], errors="coerce")
        subjectivity = reviews.groupby("App")["Sentiment_Subjectivity"].mean()
        apps["Sentiment_Subjectivity"] = apps["App"].map(subjectivity)
    else:
        apps["Sentiment_Subjectivity"] = np.nan
else:
    apps["Sentiment_Subjectivity"] = np.nan

aliases = {
    "COMMICS": "COMICS", "COMMICATION": "COMMUNICATION",
    "EVENT": "EVENTS", "GAMES": "GAME"
}
if "Category" in apps.columns:
    apps["Category"] = apps["Category"].replace(aliases)
apps = apps.drop_duplicates(subset=["App"], keep="first") if "App" in apps.columns else apps

display(Markdown(
    f"**Loaded apps:** `{apps_path.name}` — {len(apps):,} rows. "
    + (f"Reviews file: `{reviews_path.name}`." if reviews_path else
       "Reviews CSV not found; sentiment subjectivity will be unavailable.")
))
apps.head()

**Loaded apps:** `googleplaystore.csv` — 9,660 rows. Reviews file: `User_Reviews_Google_Play.csv`.

,App,Category,Rating,Reviews,Size,Installs,Type,Price,Content Rating,Genres,Last Updated,Current Ver,Android Ver,Size_MB,Sentiment_Subjectivity
0,Photo Editor & Candy Camera & Grid & ScrapBook,ART_AND_DESIGN,4.1,159.0,19M,10000.0,Free,0.0,Everyone,Art & Design,2018-01-07,1.0.0,4.0.3 and up,19.0,NaN
1,Coloring book moana,ART_AND_DESIGN,3.9,967.0,14M,500000.0,Free,0.0,Everyone,Art & Design;Pretend Play,2018-01-15,2.0.0,4.0.3 and up,14.0,NaN
2,"U Launcher Lite – FREE Live Cool Themes, Hide ...",ART_AND_DESIGN,4.7,87510.0,8.7M,5000000.0,Free,0.0,Everyone,Art & Design,2018-08-01,1.2.4,4.0.3 and up,8.7,NaN
3,Sketch - Draw & Paint,ART_AND_DESIGN,4.5,215644.0,25M,50000000.0,Free,0.0,Teen,Art & Design,2018-06-08,Varies with device,4.2 and up,25.0,NaN
4,Pixel Draw - Number Art Coloring Book,ART_AND_DESIGN,4.3,967.0,2.8M,100000.0,Free,0.0,Everyone,Art & Design;Creativity,2018-06-20,1.1,4.4 and up,2.8,NaN


## 2. Task 3 — Time Series: Monthly Installs by Category

Monthly total installs by category. Filters: app name does not start X/Y/Z; category begins E/C/B; reviews > 500; app name excludes S. Shade month-over-month growth >20%. Requested translations are included. Chart is always visible (no time restriction). Historical install data requires a real date/month field.

In [2]:
# Task 3 — no historical dates exist, so show a truthful current-snapshot category chart.
required = {"App", "Category", "Reviews", "Installs"}
missing = required - set(apps.columns)
if missing:
    display(Markdown(f"Cannot create Task 3 chart; missing columns: `{sorted(missing)}`"))
else:
    d = apps[
        ~apps["App"].str.upper().str.startswith(("X", "Y", "Z"))
        & ~apps["App"].str.contains("S", case=False, na=False)
        & (apps["Reviews"] > 500)
        & apps["Category"].str.startswith(("E", "C", "B"), na=False)
    ].copy()
    summary = d.groupby("Category", as_index=False)["Installs"].sum().sort_values("Installs", ascending=False).head(15)
    display(Markdown("**Data limitation:** This source is a single catalogue snapshot with no install date/month. Month-over-month growth cannot be calculated; the chart below shows current-snapshot installs by category instead of inventing a time series."))
    if summary.empty:
        display(Markdown("No rows match the available Task 3 app/category filters."))
    else:
        fig = px.bar(summary, x="Category", y="Installs", color="Category",
                     title="Task 3 fallback: Current-snapshot installs by eligible category")
        fig.update_layout(showlegend=False, xaxis_title="Category", yaxis_title="Total installs")
        fig.show()

**Data limitation:** This source is a single catalogue snapshot with no install date/month. Month-over-month growth cannot be calculated; the chart below shows current-snapshot installs by category instead of inventing a time series.